In [ ]:
import pandas as pd
import plotly.graph_objects as go

url = 'https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/02_filtered/Tabela_Censo_Escolar_Menor_Evasao_2024.csv'
menores_taxas = pd.read_csv(url, sep=';')

colunas_analise = [
    'IN_AGUA_POTAVEL', 'IN_AGUA_INEXISTENTE', 'IN_ENERGIA_REDE_PUBLICA',
    'IN_ENERGIA_INEXISTENTE', 'IN_ESGOTO_REDE_PUBLICA', 'IN_ESGOTO_INEXISTENTE',
    'IN_BANHEIRO', 'IN_COZINHA', 'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA',
    'IN_QUADRA_ESPORTES', 'IN_REFEITORIO', 'IN_SALA_MULTIUSO', 'IN_SALA_DIRETORIA',
    'IN_SALA_LEITURA', 'IN_SECRETARIA', 'IN_INTERNET', 'IN_INTERNET_ALUNOS', 'IN_ALIMENTACAO'
]

df_agrupado = menores_taxas.groupby('SG_UF')[colunas_analise].sum().reset_index()

fig = go.Figure()

for i, coluna in enumerate(colunas_analise):
    fig.add_trace(
        go.Bar(
            x=df_agrupado['SG_UF'],
            y=df_agrupado[coluna],
            name=coluna.replace('IN_', '').replace('_', ' '),
            visible=(i == 0), # Deixa apenas a primeira coluna visível inicialmente
            marker=dict(
                color=df_agrupado[coluna],
                colorscale=[
                    [0.0, '#5497de'],
                    [1.0, '#091f39']
                ],
                showscale=False
            )
        )
    )

total_features = len(colunas_analise)
for i, coluna in enumerate(colunas_analise):
    moda_escolas_por_estado = df_agrupado[coluna].mode()[0] if not df_agrupado[coluna].mode().empty else df_agrupado[coluna].mean()

    fig.add_trace(
        go.Scatter(
            x=df_agrupado['SG_UF'],
            y=[moda_escolas_por_estado] * len(df_agrupado),
            mode='lines',
            name=f"Moda: {coluna.replace('IN_', '').replace('_', ' ')}",
            line=dict(color='#00664b', width=2, dash='dash'),
            visible=(i == 0) 
        )
    )

botoes = []
for i, coluna in enumerate(colunas_analise):
    visibilidade = [False] * (total_features * 2)
    visibilidade[i] = True                    # Ativa a barra correspondente
    visibilidade[i + total_features] = True   # Ativa a linha de moda correspondente

    texto_botao = coluna.replace('IN_', '').replace('_', ' ').title()
    valor_moda = df_agrupado[coluna].mode()[0] if not df_agrupado[coluna].mode().empty else int(df_agrupado[coluna].mean())

    botoes.append(
        dict(
            label=texto_botao,
            method='update',
            args=[
                {'visible': visibilidade},
                {
                    'title': f'<b>Quantidade de Escolas por Estado: {texto_botao}</b><br><sub>Moda de Escolas com MENORES Taxas de Evasão por Estado: {valor_moda}</sub>'
                },
            ],
        )
    )

cor_fundo = '#F8F9FA' 

primeira_moda = df_agrupado[colunas_analise[0]].mode()[0]
fig.update_layout(
    title={
        'text': f"<b>Quantidade de Escolas por Estado: {colunas_analise[0].replace('IN_', '').replace('_', ' ').title()}</b><br><sub>Moda de Escolas com MENORES Taxas de Evasão por Estado: {primeira_moda}</sub>",
        'y': 0.89,
        'x': 0.45,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(color='#111111')
    },
    xaxis_title='<b>Estado (UF)</b>',
    yaxis_title='<b>Quantidade de Escolas</b>',

    plot_bgcolor=cor_fundo,
    paper_bgcolor=cor_fundo,

    xaxis=dict(showgrid=True, gridcolor='#EBEBEB'),
    yaxis=dict(showgrid=True, gridcolor='#EBEBEB'),

    margin=dict(t=150, b=50, l=50, r=50),

    updatemenus=[
        dict(
            active=0,
            buttons=botoes,
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.5,
            xanchor='center',
            y=1.20,
            yanchor='top',
            bgcolor='white',        # Fundo do botão branco para destacar do cinza
            bordercolor='#CCCCCC',  # Borda suave no botão
            font=dict(color='#333333')
        )
    ],
)

fig.show()